# Local Modeling Demo from Raw Kaggle Data

This notebook starts from `data/train.csv` and `data/test.csv`, builds the modeling features, and recomputes the local modeling evidence directly. It does not read saved experiment result tables.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "data" / "train.csv").exists() and (path / "data" / "test.csv").exists():
            return path
    raise FileNotFoundError("Open this notebook from inside the repository folder.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROJECT_ROOT

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")
display(train_raw.head())
pd.DataFrame({"split": ["train", "test"], "rows": [len(train_raw), len(test_raw)], "columns": [train_raw.shape[1], test_raw.shape[1]]})

## Feature Engineering

The engineered matrix is still built only from official competition columns: passenger groups, cabin structure, route, spending behavior, and cryosleep-spend consistency.

In [ ]:
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    group_split = df["PassengerId"].str.split("_", expand=True)
    df["PassengerGroup"] = group_split[0]
    df["PassengerNo"] = pd.to_numeric(group_split[1], errors="coerce")
    df["GroupSize"] = df.groupby("PassengerGroup")["PassengerId"].transform("count")
    df["IsAlone"] = (df["GroupSize"] == 1).astype(int)

    cabin_split = df["Cabin"].fillna("Unknown/Unknown/Unknown").str.split("/", expand=True)
    df["Deck"] = cabin_split[0]
    df["CabinNum"] = pd.to_numeric(cabin_split[1].replace("Unknown", np.nan), errors="coerce")
    df["Side"] = cabin_split[2]
    df["DeckSide"] = df["Deck"].astype(str) + "_" + df["Side"].astype(str)

    for col in SPEND_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["TotalSpend"] = df[SPEND_COLS].fillna(0).sum(axis=1)
    df["LogTotalSpend"] = np.log1p(df["TotalSpend"])
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)
    df["LuxurySpend"] = df[["Spa", "VRDeck", "FoodCourt"]].fillna(0).sum(axis=1)
    df["BasicSpend"] = df[["RoomService", "ShoppingMall"]].fillna(0).sum(axis=1)
    df["NonZeroSpendCount"] = (df[SPEND_COLS].fillna(0) > 0).sum(axis=1)

    df["AgeBand"] = pd.cut(
        df["Age"],
        bins=[-1, 12, 18, 25, 40, 60, 200],
        labels=["child", "teen", "young_adult", "adult", "middle_age", "senior"],
    ).astype("object")
    df["Route"] = df["HomePlanet"].fillna("Unknown").astype(str) + "_" + df["Destination"].fillna("Unknown").astype(str)
    df["CryoNoSpendMatch"] = (df["CryoSleep"].fillna(False).astype(bool) & df["NoSpend"].eq(1)).astype(int)
    df["AgeMissing"] = df["Age"].isna().astype(int)
    df["CabinKnown"] = df["Cabin"].notna().astype(int)
    return df

train = add_features(train_raw)
test = add_features(test_raw)

feature_families = {
    "official": ["HomePlanet", "CryoSleep", "Destination", "Age", "VIP", *SPEND_COLS],
    "cabin": ["Deck", "CabinNum", "Side", "DeckSide", "CabinKnown"],
    "group": ["PassengerGroup", "PassengerNo", "GroupSize", "IsAlone"],
    "spending": ["TotalSpend", "LogTotalSpend", "NoSpend", "LuxurySpend", "BasicSpend", "NonZeroSpendCount"],
    "consistency": ["AgeBand", "Route", "CryoNoSpendMatch", "AgeMissing"],
}
features = [col for cols in feature_families.values() for col in cols]
numeric_features = [col for col in features if pd.api.types.is_numeric_dtype(train[col])]
categorical_features = [col for col in features if col not in numeric_features]

display(pd.DataFrame({"family": feature_families.keys(), "n_features": [len(v) for v in feature_families.values()], "features": [", ".join(v) for v in feature_families.values()]}))
train[features + ["Transported"]].head()

## Modeling Protocol

The classroom run uses group-aware cross-validation by the `PassengerId` prefix. This is the same independence idea used in the report: passengers from the same travel group stay in the same fold.

In [ ]:
N_SPLITS = 5
SEED = 42
X = train[features]
y = train["Transported"].astype(int)
groups = train["PassengerGroup"]

def make_preprocessor(feature_subset: list[str]) -> ColumnTransformer:
    num_cols = [col for col in feature_subset if col in numeric_features]
    cat_cols = [col for col in feature_subset if col in categorical_features]
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), num_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), cat_cols),
        ]
    )

def evaluate_model(model_name: str, model, feature_subset: list[str]) -> dict:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, valid_idx in splitter.split(train[feature_subset], y, groups):
        pipeline = Pipeline([("preprocess", make_preprocessor(feature_subset)), ("model", model)])
        pipeline.fit(train.iloc[train_idx][feature_subset], y.iloc[train_idx])
        pred = pipeline.predict(train.iloc[valid_idx][feature_subset])
        scores.append(accuracy_score(y.iloc[valid_idx], pred))
    return {
        "model": model_name,
        "cv_accuracy_mean": float(np.mean(scores)),
        "cv_accuracy_std": float(np.std(scores)),
        "fold_scores": [round(score, 5) for score in scores],
    }

pd.DataFrame([
    ["CV scheme", "5-fold StratifiedGroupKFold"],
    ["Group key", "PassengerId prefix"],
    ["Threshold", "0.5"],
    ["Input source", "data/train.csv only"],
], columns=["item", "value"])

## Model Comparison

This table is generated by fitting each model during the notebook run.

In [ ]:
models = [
    ("Logistic Regression", Pipeline([("scale", StandardScaler()), ("clf", LogisticRegression(max_iter=1500, random_state=SEED))])),
    ("Random Forest", RandomForestClassifier(n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=SEED)),
    ("LightGBM", LGBMClassifier(n_estimators=350, learning_rate=0.035, num_leaves=31, subsample=0.9, colsample_bytree=0.9, random_state=SEED, verbosity=-1)),
    ("XGBoost", XGBClassifier(n_estimators=350, learning_rate=0.035, max_depth=5, subsample=0.9, colsample_bytree=0.85, random_state=SEED, eval_metric="logloss", tree_method="hist", n_jobs=-1)),
    ("CatBoost", CatBoostClassifier(iterations=350, learning_rate=0.035, depth=6, loss_function="Logloss", random_seed=SEED, verbose=False, thread_count=-1)),
]

model_results = pd.DataFrame([evaluate_model(name, model, features) for name, model in models])
model_results.sort_values("cv_accuracy_mean", ascending=False).reset_index(drop=True)

## Feature-Family Ablation

The same XGBoost setup is rerun while removing one feature family at a time. This shows which modeling data carries signal.

In [ ]:
def without_family(family: str) -> list[str]:
    remove = set(feature_families[family])
    return [col for col in features if col not in remove]

ablation_rows = []
ablation_model = XGBClassifier(n_estimators=300, learning_rate=0.04, max_depth=5, subsample=0.9, colsample_bytree=0.85, random_state=SEED, eval_metric="logloss", tree_method="hist", n_jobs=-1)
full_result = evaluate_model("all_features", ablation_model, features)
ablation_rows.append({"feature_set": "all_features", "n_features": len(features), **full_result})

for family in ["cabin", "group", "spending", "consistency"]:
    subset = without_family(family)
    result = evaluate_model(f"without_{family}", ablation_model, subset)
    ablation_rows.append({"feature_set": f"without_{family}", "n_features": len(subset), **result})

ablation_results = pd.DataFrame(ablation_rows)
baseline = ablation_results.loc[ablation_results["feature_set"].eq("all_features"), "cv_accuracy_mean"].iloc[0]
ablation_results["delta_vs_all"] = ablation_results["cv_accuracy_mean"] - baseline
ablation_results[["feature_set", "n_features", "cv_accuracy_mean", "cv_accuracy_std", "delta_vs_all", "fold_scores"]]

## Small XGBoost Neighborhood Search

This is a compact local tuning demonstration. It is not the Kaggle 0.81716 reproduction; that path is kept in the Kaggle notebook.

In [ ]:
xgb_configs = [
    {"label": "baseline", "n_estimators": 250, "learning_rate": 0.045, "max_depth": 4, "subsample": 0.90, "colsample_bytree": 0.85, "reg_alpha": 0.0, "reg_lambda": 1.0},
    {"label": "more_trees", "n_estimators": 400, "learning_rate": 0.030, "max_depth": 4, "subsample": 0.90, "colsample_bytree": 0.85, "reg_alpha": 0.0, "reg_lambda": 1.0},
    {"label": "deeper_regularized", "n_estimators": 350, "learning_rate": 0.035, "max_depth": 5, "subsample": 0.90, "colsample_bytree": 0.85, "reg_alpha": 0.1, "reg_lambda": 1.5},
    {"label": "sparser_sampling", "n_estimators": 350, "learning_rate": 0.035, "max_depth": 5, "subsample": 0.80, "colsample_bytree": 0.75, "reg_alpha": 0.1, "reg_lambda": 1.5},
]

tuning_rows = []
for cfg in xgb_configs:
    params = {k: v for k, v in cfg.items() if k != "label"}
    model = XGBClassifier(**params, random_state=SEED, eval_metric="logloss", tree_method="hist", n_jobs=-1)
    result = evaluate_model(cfg["label"], model, features)
    tuning_rows.append({**cfg, "cv_accuracy_mean": result["cv_accuracy_mean"], "cv_accuracy_std": result["cv_accuracy_std"], "fold_scores": result["fold_scores"]})

pd.DataFrame(tuning_rows).sort_values("cv_accuracy_mean", ascending=False).reset_index(drop=True)